# Lab 4-2: Load Data

Author: Seungjae Lee (이승재)  
GitHub의 자료와 실제 강의 내용이 일치하지 않아 자료를 수정했습니다. (이지원)

<div class="alert alert-warning">
    We use elemental PyTorch to implement linear regression here. However, in most actual applications, abstractions such as <code>nn.Module</code> or <code>nn.Linear</code> are used.
</div>

## Imports

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# For reproducibility
torch.manual_seed(1)

## MInibatch
복잡한 머신러닝 모델을 학습하기 위해 엄청난 양의 데이터가 필요하다.  
하지만 대용량의 데이터를 한 번에 학습시키기에는 하드웨어적인 장애물이 존재한다.  
-> 일부분의 데이터로만 학습하는 건 어떨까? 전체 데이터를 균일하게 나눠서 학습시키자!

## CustomDataset 정의


In [7]:
from torch.utils.data import Dataset  # 데이터 관리를 위한 기본 클래스

class CustomDataset(Dataset):
    def __init__(self):
        self.x_data = [
            [73, 80, 75],
            [93, 88, 93],
            [89, 91, 90],
            [96, 98, 100],
            [73, 66, 70]]

        self.y_data = [
            [152],
            [185],
            [180],
            [196],
            [142]]

    def __len__(self):                     # 매직메서드 len, getitem 필요
        return len(self.x_data)

    def __getitem__(self, idx):
        x = torch.FloatTensor(self.x_data[idx])
        y = torch.FloatTensor(self.y_data[idx])

        return x, y


dataset = CustomDataset()



## DataLoader 생성
Dataset에 저장된 데이터를 써내서 모델에게 전달해준다.

In [6]:
from torch.utils.data import DataLoader  # Dataset에서 데이터를 읽어오는 클래스

dataloader = DataLoader(  # 객체 정의
    dataset,  
    batch_size=2,  # 데이터를 한 번에 두 개씩 모델에게 전달하겠다. 
    shuffle=True,  # 매 epoch마다 데이터 순서를 랜덤하게 섞는다. (학습 편향 방지)
)

## Full Code

In [9]:
class MultivariateLinearRegressionModel(nn.Module):  # 새로운 선형회귀 모델 클래스를 정의.
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1)  # 입력 3, 출력 1

    def forward(self, x):
        return self.linear(x)
    
x_train = torch.FloatTensor([[73, 80, 75],
                             [93, 88, 93],
                             [89, 91, 90],
                             [96, 98, 100],
                             [73, 66, 70]])
y_train = torch.FloatTensor([[152], [185], [180], [196], [142]])
# 모델 초기화
model = MultivariateLinearRegressionModel()
# optimizer 설정
optimizer = optim.SGD(model.parameters(), lr=1e-5)  # model.parameters(): W, b를 자동으로 찾아 optimizer에 전달하는 함수.


nb_epochs = 20

for epoch in range(nb_epochs + 1):
    for batch_idx, samples in enumerate(dataloader):  # enurmerate를 통한 index와 각 batch 값이 batch_idx, samples에 나누어 들어감.
        x_train, y_train = samples

        # H(x) 계산
        prediction = model(x_train)

        # cost 계산
        cost = F.mse_loss(prediction, y_train)

        # cost로 H(x) 개선
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        print(
            'Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
                epoch,
                nb_epochs,
                batch_idx + 1,
                len(dataloader),
                cost.item()
            )
        )

Epoch    0/20 Batch 1/3 Cost: 35103.968750
Epoch    0/20 Batch 2/3 Cost: 5925.153809
Epoch    0/20 Batch 3/3 Cost: 4785.246582
Epoch    1/20 Batch 1/3 Cost: 825.066101
Epoch    1/20 Batch 2/3 Cost: 140.155396
Epoch    1/20 Batch 3/3 Cost: 37.768578
Epoch    2/20 Batch 1/3 Cost: 27.472120
Epoch    2/20 Batch 2/3 Cost: 4.423943
Epoch    2/20 Batch 3/3 Cost: 4.016191
Epoch    3/20 Batch 1/3 Cost: 0.879160
Epoch    3/20 Batch 2/3 Cost: 0.490072
Epoch    3/20 Batch 3/3 Cost: 0.192556
Epoch    4/20 Batch 1/3 Cost: 0.658972
Epoch    4/20 Batch 2/3 Cost: 0.186390
Epoch    4/20 Batch 3/3 Cost: 0.264580
Epoch    5/20 Batch 1/3 Cost: 0.666050
Epoch    5/20 Batch 2/3 Cost: 0.227237
Epoch    5/20 Batch 3/3 Cost: 0.281740
Epoch    6/20 Batch 1/3 Cost: 0.395982
Epoch    6/20 Batch 2/3 Cost: 0.407762
Epoch    6/20 Batch 3/3 Cost: 0.249421
Epoch    7/20 Batch 1/3 Cost: 0.348882
Epoch    7/20 Batch 2/3 Cost: 0.282357
Epoch    7/20 Batch 3/3 Cost: 0.889984
Epoch    8/20 Batch 1/3 Cost: 0.070224
Epoch    